In [ ]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# === ПУТИ К ТВОИМ ФАЙЛАМ ===
paths = [
    '/Users/dmitrii/Downloads/cleaned_Автосервис ИТС.csv',
    '/Users/dmitrii/Downloads/cleaned_Мойка.csv'
]
out_dir = '/Users/dmitrii/Downloads/visuals_отчет'
os.makedirs(out_dir, exist_ok=True)

# === ЗАГРУЗКА ДАННЫХ ===
dfs = {}
for p in paths:
    name = os.path.basename(p)
    df = pd.read_csv(p)
    df['date'] = pd.to_datetime(df['По дням'])
    df = df.set_index('date').sort_index()
    df['dow'] = df.index.dayofweek
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['week'] = df.index.isocalendar().week.astype(int)
    dfs[name] = df

# === ВИЗУАЛИЗАЦИИ ===
for name, df in dfs.items():
    title = name.replace('cleaned_', '').replace('.csv', '')
    for col in ['Количество заказ-нарядов','Всего по заказ-наряду','Работ','Запчастей']:
        if col not in df.columns:
            continue
        
        # Линейный график
        plt.figure(figsize=(10,4))
        df[col].plot()
        plt.title(f'{title} — {col}')
        plt.tight_layout()
        plt.savefig(f'{out_dir}/{title}_{col}_line.png', dpi=160)
        plt.close()

        # Boxplot по дням недели
        plt.figure(figsize=(8,4))
        plt.boxplot([df.loc[df.dow==i, col].dropna() for i in range(7)],
                    labels=['Пн','Вт','Ср','Чт','Пт','Сб','Вс'], showmeans=True)
        plt.title(f'{title} — {col} по дням недели')
        plt.tight_layout()
        plt.savefig(f'{out_dir}/{title}_{col}_box_dow.png', dpi=160)
        plt.close()

        # Boxplot по месяцам
        plt.figure(figsize=(9,4))
        plt.boxplot([df.loc[df.month==i, col].dropna() for i in range(1,13)],
                    labels=range(1,13), showmeans=True)
        plt.title(f'{title} — {col} по месяцам')
        plt.tight_layout()
        plt.savefig(f'{out_dir}/{title}_{col}_box_month.png', dpi=160)
        plt.close()

        # Heatmap (Год × Неделя)
        pivot = pd.pivot_table(df, values=col, index='year', columns='week', aggfunc='mean')
        plt.figure(figsize=(12,4))
        plt.imshow(pivot.fillna(0), aspect='auto')
        plt.title(f'{title} — {col}: heatmap (Год × Неделя)')
        plt.xlabel('Неделя'); plt.ylabel('Год')
        plt.colorbar(label=col)
        plt.tight_layout()
        plt.savefig(f'{out_dir}/{title}_{col}_heat_year_week.png', dpi=160)
        plt.close()

        # STL-разложение
        stl = STL(df[col].interpolate(limit_direction='both'), period=7, robust=True).fit()
        components = {'trend': stl.trend, 'seasonal': stl.seasonal, 'resid': stl.resid}
        for comp_name, series in components.items():
            plt.figure(figsize=(10,2.5))
            series.plot()
            plt.title(f'{title} — {col}: {comp_name}')
            plt.tight_layout()
            plt.savefig(f'{out_dir}/{title}_{col}_stl_{comp_name}.png', dpi=160)
            plt.close()

        # ACF / PACF
        fig, ax = plt.subplots(1,2, figsize=(10,3))
        plot_acf(df[col].dropna(), lags=60, ax=ax[0])
        plot_pacf(df[col].dropna(), lags=60, ax=ax[1])
        ax[0].set_title('ACF'); ax[1].set_title('PACF')
        plt.suptitle(f'{title} — {col}')
        plt.tight_layout()
        plt.savefig(f'{out_dir}/{title}_{col}_acf_pacf.png', dpi=160)
        plt.close()

# === ПРОСТОЙ АНАЛИЗ И ОТЧЁТ ===
insights = []
for name, df in dfs.items():
    base = name.replace('cleaned_', '').replace('.csv','')
    for col in ['Всего по заказ-наряду','Количество заказ-нарядов']:
        if col not in df.columns:
            continue
        y = df[col].values.reshape(-1,1)
        x = np.arange(len(df)).reshape(-1,1)
        slope = LinearRegression().fit(x,y).coef_[0][0]
        direction = "рост" if slope > 0 else "снижение" if slope < 0 else "стабильность"
        insights.append(f"**{base} — {col}**: тренд = {direction}, наклон = {slope:.3f}")

# === СОЗДАНИЕ MARKDOWN-ОТЧЁТА ===
report_path = os.path.join(out_dir, 'Отчёт_временные_ряды_локально.md')
images = sorted([f for f in os.listdir(out_dir) if f.endswith('.png')])

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(f"# Аналитический отчёт по временным рядам\nСформировано: {datetime.now()}\n\n")
    f.write("## Ключевые наблюдения\n" + "\n".join(["- " + i for i in insights]) + "\n\n---\n# 📊 Визуализации\n")
    for img in images:
        f.write(f"\n### {img.replace('_', ' ').replace('.png','')}\n![]({out_dir}/{img})\n")

print(f"✅ Готово! Отчёт сохранён сюда:\n{report_path}")
